<a href="https://colab.research.google.com/github/saikttech/colab-aiagent/blob/main/%D0%94%D0%97_Pro_%7C_%D0%9C%D0%B5%D1%82%D0%BE%D0%B4%D1%8B_Faiss%2C_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Задание Pro к занятию Методы Faiss, Langchain
Возьмите несколько (от двух и более) любых текстовых документов, на основании информации из которых вы создадите консультанта.

Можете взять две части баз знаний компании Simble (без разбивки MarkDown). Ссылки даны ниже.

url_base1_simble: https://docs.google.com/document/d/1Z7eZLIPG9URgOFz-yqtJAup-WXhKFIiF

url_base2_simble: https://docs.google.com/document/d/1qxJXwHtYNxx6ecf35zhqFYBxSjoA5Mhr

Разбейте каждый текст на чанки любым удобным и подходящим способом
Создайте отдельные векторные базы на каждую часть и сохраниете их на ГуглДиск
Загрузите из ГуглДиска сохраненные на предыдущем шаге отдельные векторные базы и объедините их в одну.
Создайте простого консультанта по объединенной базе знаний. Используйте ретривер и ChatOpenAI из Langchain.
Задайте вопрос консультанту по базе знаний и получите ответ


# Решение

In [ ]:
# =============================================================================
# Задание Pro: Методы FAISS, Langchain — Консультант по базе знаний Simble
# Используем LCEL (LangChain Expression Language)
# =============================================================================

# --- 1. Установка необходимых библиотек ---
!pip install -q --upgrade langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu openai ipywidgets

import os
import requests
from google.colab import userdata, drive

# --- ИМПОРТЫ ---
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import ipywidgets as widgets
from ipywidgets import Dropdown, Button, Output, VBox

# --- 2. Подключение Google Drive ---
drive.mount('/content/drive')
DRIVE_PATH = "/content/drive/MyDrive/faiss_simble"
os.makedirs(DRIVE_PATH, exist_ok=True)

# --- 3. Получение API-ключа OpenAI из секретов Colab ---
try:
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except userdata.SecretNotFoundError:
    raise RuntimeError(
        "❗ API-ключ не найден. Добавьте его в Colab Secrets "
        "(иконка ключа слева) с именем OPENAI_API_KEY"
    )
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# --- 4. Загрузка текстовых документов с Google Docs ---
DOC_IDS = {
    "base1": "1Z7eZLIPG9URgOFz-yqtJAup-WXhKFIiF",
    "base2": "1qxJXwHtYNxx6ecf35zhqFYBxSjoA5Mhr",
}

def download_google_doc(doc_id: str) -> str:
    """Скачивает публичный Google Doc в формате plain text."""
    url = f"https://docs.google.com/document/d/{doc_id}/export?format=txt"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text

print("📥 Загружаем документы...")
text_base1 = download_google_doc(DOC_IDS["base1"])
text_base2 = download_google_doc(DOC_IDS["base2"])
print(f"✅ База 1: {len(text_base1)} символов")
print(f"✅ База 2: {len(text_base2)} символов")

# --- 5. Разбиение текстов на чанки ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks_base1 = text_splitter.create_documents([text_base1])
chunks_base2 = text_splitter.create_documents([text_base2])

for doc in chunks_base1:
    doc.metadata["source"] = "base1_simble"
for doc in chunks_base2:
    doc.metadata["source"] = "base2_simble"

print(f"📦 База 1 разбита на {len(chunks_base1)} чанков")
print(f"📦 База 2 разбита на {len(chunks_base2)} чанков")

# --- 6. Создание отдельных векторных баз FAISS ---
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("🧠 Создаём векторные базы...")
faiss_base1 = FAISS.from_documents(chunks_base1, embeddings)
faiss_base2 = FAISS.from_documents(chunks_base2, embeddings)

print(f"📊 База 1: {faiss_base1.index.ntotal} векторов")
print(f"📊 База 2: {faiss_base2.index.ntotal} векторов")

# --- 7. Сохранение отдельных баз на Google Drive ---
PATH_BASE1 = os.path.join(DRIVE_PATH, "faiss_base1")
PATH_BASE2 = os.path.join(DRIVE_PATH, "faiss_base2")

faiss_base1.save_local(PATH_BASE1)
faiss_base2.save_local(PATH_BASE2)
print(f"💾 Базы сохранены на Google Drive: {DRIVE_PATH}")

# --- 8. Загрузка сохранённых баз с Google Drive ---
print("🔄 Загружаем базы обратно с Google Drive...")
loaded_base1 = FAISS.load_local(
    PATH_BASE1, embeddings, allow_dangerous_deserialization=True
)
loaded_base2 = FAISS.load_local(
    PATH_BASE2, embeddings, allow_dangerous_deserialization=True
)
print(f"✅ Загружено: {loaded_base1.index.ntotal} + {loaded_base2.index.ntotal} векторов")

# --- 9. Объединение двух баз в одну ---
loaded_base1.merge_from(loaded_base2)
merged_db = loaded_base1
print(f"🔗 Объединённая база: {merged_db.index.ntotal} векторов")

# --- 10. Создание консультанта с использованием LCEL ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Создаём промпт
prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты — вежливый и компетентный консультант сервиса Simble.
Отвечай ТОЛЬКО на основе информации из контекста ниже.
Если ответа в контексте нет — честно скажи, что не знаешь.

Контекст:
{context}"""),
    ("human", "{input}"),
])

# Функция для форматирования документов
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Создаём ретривер
retriever = merged_db.as_retriever(search_kwargs={"k": 4})

# Создаём цепочку с использованием LCEL (LangChain Expression Language)
rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# --- 11. Интерактивный виджет для вопросов ---
output = Output()

question_options = [
    "Какие тарифы страхования предлагает Simble?",
    "Кто является страховым партнёром Simble?",
    "Что покрывает полис КАСКО в Simble?",
    "Какие контакты службы поддержки Simble?",
    "Что такое франшиза в полисе Simble?",
]

dropdown = Dropdown(
    options=question_options,
    description="Вопрос:",
    layout=widgets.Layout(width="70%"),
)
text_input = widgets.Text(
    placeholder="Или введите свой вопрос...",
    description="Свой:",
    layout=widgets.Layout(width="70%"),
)
btn = widgets.Button(description="Спросить консультанта", button_style="primary")

def on_button_clicked(_):
    question = text_input.value.strip() or dropdown.value
    with output:
        output.clear_output()
        print(f"\n❓ Вопрос: {question}\n")

        # Получаем ответ
        answer = rag_chain.invoke(question)
        print(f"🤖 Ответ консультанта:\n{answer}\n")

        # Получаем источники
        docs = retriever.invoke(question)
        print("📚 Источники:")
        for i, doc in enumerate(docs, 1):
            src = doc.metadata.get("source", "—")
            preview = doc.page_content[:150].replace("\n", " ")
            print(f"  [{i}] ({src}) {preview}...")

btn.on_click(on_button_clicked)
display(VBox([dropdown, text_input, btn, output]))